# Ensemble member divergence
## Why did one bootstrapped member's training round diverge, and could bootstrap variance alone explain it?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cascade.agents.db_orm import TrajectoryDB, DBTrainingFrame

In [ ]:
db_url = 'postgresql://ase:pw@localhost:5432/cascade'
db = TrajectoryDB(db_url)
run_id = '2026.08.24-18:12:33-ffdfbd'
training_round = 1  # the round where member 2's loss diverged

## Per-member provenance
`Trainer.train_model` draws each member's bootstrap sample with an unseeded `np.random.default_rng()` and only ever holds the resulting indices in memory (`cascade/agents/agents.py`) -- they're never written to the DB or logs. So there is no way to recover which exact frames (with duplicates) went to which of the 4 members for this run. What follows instead: characterize the actual round's training pool, then simulate the real resampling procedure on that real pool to see how much per-member variance it produces on its own.

In [ ]:
def get_round_physical_metrics(db, run_id, training_round):
    """Physical-absurdity metrics per training frame, computed directly from the labeled
    (reference) Atoms -- independent of any model prediction."""
    with db.session() as sess:
        rows = sess.query(DBTrainingFrame).filter_by(run_id=run_id, training_round=training_round).all()
        records = []
        for tf in rows:
            atoms_labeled = db._deserialize_atoms(tf.atoms_labeled_blob)
            forces = atoms_labeled.calc.results['forces']
            dists = atoms_labeled.get_all_distances(mic=True)
            np.fill_diagonal(dists, np.inf)
            records.append({
                'traj_id': tf.traj_id,
                'chunk_id': tf.chunk_id,
                'calibration_error': tf.calibration_error,
                'max_ref_force': float(np.linalg.norm(forces, axis=-1).max()),
                'cell_volume': atoms_labeled.get_volume(),
                'min_interatomic_dist': float(dists.min()),
            })
    return pd.DataFrame(records)

pool_df = get_round_physical_metrics(db, run_id, training_round)
pool_df.shape

In [ ]:
pool_df[['calibration_error', 'max_ref_force', 'cell_volume', 'min_interatomic_dist']].describe(
    percentiles=[.05, .25, .5, .75, .9, .95, .99]
)

In [ ]:
for threshold in [1.0, 0.5, 0.2]:
    frac = (pool_df['min_interatomic_dist'] < threshold).mean()
    print(f'fraction of round-{training_round} frames with min interatomic distance < {threshold} A: {frac:.1%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(pool_df['min_interatomic_dist'], bins=30)
axes[0].set_xlabel('min interatomic distance ($\\mathrm{\\AA}$)')
axes[0].set_ylabel('count')
axes[1].hist(pool_df['max_ref_force'], bins=30)
axes[1].set_xlabel('max reference force ($\\mathrm{eV/\\AA}$)')
axes[1].set_ylabel('count')
fig.suptitle(f'Round {training_round} training pool, n={len(pool_df)}')
plt.tight_layout()
plt.show()

## Monte Carlo: does bootstrap variance alone explain one member drawing a worse hand than the others?
Simulates the actual resampling procedure (`bootstrap_fraction=1.0` -> sample-with-replacement, `n_sample == len(pool)`, see `cascade/agents/config.py` / `Trainer.train_model`) on the real round's `max_ref_force` values, many times, for 4 independent members per trial.

In [ ]:
bootstrap_fraction = 1.0  # matches TrainerConfig default
n_members = 4
n_trials = 5000

vals = pool_df['max_ref_force'].values
n_sample = int(len(vals) * bootstrap_fraction)
worst_idx = int(np.argmax(vals))

rng = np.random.default_rng(0)  # seeded only so this simulation itself is reproducible -- the real run's draw was not seeded and is not recoverable
draws = rng.integers(0, len(vals), size=(n_trials, n_members, n_sample))

worst_frame_copies = (draws == worst_idx).sum(axis=2)  # (n_trials, n_members): copies of the single worst frame each member happened to draw
exposure = (vals[draws] ** 2).sum(axis=2)  # (n_trials, n_members): total force^2 "gradient shock" exposure per member's draw

copy_spread = worst_frame_copies.max(axis=1) - worst_frame_copies.min(axis=1)
exposure_ratio = exposure.max(axis=1) / exposure.min(axis=1)

print(f'worst single frame in the real pool: max_ref_force={vals[worst_idx]:.1f} eV/A')
print(f'P(most-exposed member draws >=2 more copies of it than least-exposed member): {(copy_spread >= 2).mean():.1%}')
print(f'P(most-exposed member draws >=3 more copies of it than least-exposed member): {(copy_spread >= 3).mean():.1%}')
print()
print('per-trial (max member exposure / min member exposure) ratio:')
print(pd.Series(exposure_ratio).describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(worst_frame_copies.flatten(), bins=np.arange(worst_frame_copies.max() + 2) - 0.5)
axes[0].set_xlabel('copies of the single worst frame in one member\'s bootstrap draw')
axes[0].set_ylabel('count (across all simulated members)')
axes[1].hist(exposure_ratio, bins=30)
axes[1].set_xlabel('per-trial max/min member force$^2$ exposure ratio')
axes[1].set_ylabel('count (across trials)')
fig.suptitle('Bootstrap-variance simulation, 4 members x 5000 trials')
plt.tight_layout()
plt.show()